In [9]:
import os, json, time
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/fever_project'
except ImportError:
    ROOT = './fever_project'          # running outside Colab
if not os.path.isdir(ROOT):
    raise FileNotFoundError(f'{ROOT} not found. Add the shared fever_project folder as a shortcut in My Drive.')
for d in ['data/processed', 'results']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

LABELS   = ['SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO']   # fixed order for everyone
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
CONTRACT_COLS = ['id', 'claim', 'evidence', 'label', 'label_id', 'evidence_source', 'n_sent']

def check_contract(df, name='dataframe'):
    """Fail loudly if a table does not follow the team's data contract (same check as Member 3)."""
    missing = [c for c in CONTRACT_COLS if c not in df.columns]
    assert not missing, f'{name}: missing columns {missing}'
    assert df['id'].is_unique, f'{name}: duplicate ids'
    assert df['label'].isin(LABELS).all(), f'{name}: unexpected label strings'
    assert (df['label'].map(label2id) == df['label_id']).all(), f'{name}: label_id does not match label'
    assert df['claim'].str.len().gt(0).all() and df['evidence'].str.len().gt(0).all(), f'{name}: empty claim or evidence'
    print(f'{name}: contract OK ({len(df)} rows)')

MINI = False                      # True = quick test on a sample, False = real data
MINI_GOLD_N = 4000               # gold train claims used in MINI mode (validation uses a quarter of this)
SEED = 42
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128                    # must equal the training max length (06_train, 07_evaluate_errors)
# NEI claims in TRAIN whose exact text is also a SUPPORTS/REFUTES claim give the same text two labels.
DROP_NEI_TEXT_CONFLICTS_TRAIN = False
suffix = '_mini' if MINI else ''
print('ROOT:', ROOT, '| MINI:', MINI, '| MAX_LEN:', MAX_LEN)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT: /content/drive/MyDrive/fever_project | MINI: False | MAX_LEN: 128


In [10]:
P = f'{ROOT}/data/processed'
need = [f'{P}/gold_train.parquet', f'{P}/gold_val.parquet', f'{P}/nei_train{suffix}.parquet', f'{P}/nei_val{suffix}.parquet']
for p in need:
    assert os.path.exists(p), f'Missing {p}. Run 04_build_nei_tfidf (MINI = {MINI}) first, or ask Member 1 for the gold files.'

gold_train = pd.read_parquet(need[0]); gold_val = pd.read_parquet(need[1])
nei_train  = pd.read_parquet(need[2]); nei_val  = pd.read_parquet(need[3])

BASE = ['id', 'claim', 'evidence', 'label', 'n_sent', 'evidence_source']
for n, d in [('gold_train', gold_train), ('gold_val', gold_val), ('nei_train', nei_train), ('nei_val', nei_val)]:
    miss = [c for c in BASE if c not in d.columns]
    assert not miss, f'{n}: missing columns {miss}'
assert set(gold_train.label) <= {'SUPPORTS', 'REFUTES'} and set(nei_train.label) == {'NOT ENOUGH INFO'}

if MINI:   # quick pipeline test: sample the gold side as well
    gold_train = gold_train.sample(min(MINI_GOLD_N, len(gold_train)), random_state=SEED)
    gold_val = gold_val.sample(min(MINI_GOLD_N // 4, len(gold_val)), random_state=SEED)

def count_claims(split):
    p = f'{P}/claims_{split}_flat.parquet'
    if os.path.exists(p):
        d = pd.read_parquet(p, columns=['id', 'label'])
        return d.id.nunique(), d[d.label == 'NOT ENOUGH INFO'].id.nunique()
    return None, None

claims_total, nei_total = {}, {}
for split in ('train', 'val'):
    claims_total[split], nei_total[split] = count_claims(split)

print('gold   train:', len(gold_train), '| val:', len(gold_val))
print('NEI    train:', len(nei_train), '| val:', len(nei_val))
for split, nei in (('train', nei_train), ('val', nei_val)):
    if nei_total[split] is not None and not MINI:
        print(f'NEI claims {split}: {nei_total[split]} in the raw data -> {len(nei)} kept '
              f'({nei_total[split] - len(nei)} dropped by 04: no similar sentence found)')

gold   train: 109810 | val: 6666
NEI    train: 35638 | val: 3331
NEI claims train: 35639 in the raw data -> 35638 kept (1 dropped by 04: no similar sentence found)
NEI claims val: 3333 in the raw data -> 3331 kept (2 dropped by 04: no similar sentence found)


In [11]:
# (a) NEI claims whose exact text is also a SUPPORTS/REFUTES claim in the same split
sr_text_train = set(gold_train.claim); sr_text_val = set(gold_val.claim)
conf_train = nei_train.claim.isin(sr_text_train)
conf_val = nei_val.claim.isin(sr_text_val)
print(f'NEI claims with the same text as a SUPPORTS/REFUTES claim   train: {int(conf_train.sum())} of {len(nei_train)} '
      f'({conf_train.mean():.1%}) | val: {int(conf_val.sum())} of {len(nei_val)} ({conf_val.mean():.1%})')

n_dropped_conflict = 0
if DROP_NEI_TEXT_CONFLICTS_TRAIN:
    n_dropped_conflict = int(conf_train.sum())
    nei_train = nei_train[~conf_train].reset_index(drop=True)
    print(f'DROPPED {n_dropped_conflict} conflicting NEI claims from TRAIN. Validation is left untouched.')
else:
    print('Nothing dropped (DROP_NEI_TEXT_CONFLICTS_TRAIN = False). The group decides.')

NEI claims with the same text as a SUPPORTS/REFUTES claim   train: 1101 of 35638 (3.1%) | val: 35 of 3331 (1.1%)
Nothing dropped (DROP_NEI_TEXT_CONFLICTS_TRAIN = False). The group decides.


In [12]:
def finalize(gold, nei, shuffle, seed=42):
    df = pd.concat([gold[BASE], nei[BASE]], ignore_index=True)
    # same whitespace everywhere (gold passages were joined with a newline, NEI with a space)
    df['evidence'] = df['evidence'].str.replace(r'\s+', ' ', regex=True).str.strip()
    df['claim'] = df['claim'].str.replace(r'\s+', ' ', regex=True).str.strip()
    df['label_id'] = df['label'].map(label2id)
    assert df['label_id'].notna().all(), 'unknown label string'
    assert df['id'].is_unique, 'duplicate claim ids after the merge (gold and NEI share an id?)'
    assert (df['evidence'].str.len() > 0).all(), 'empty evidence'
    df['label_id'] = df['label_id'].astype(int)
    df = df[CONTRACT_COLS]
    # train: shuffled once, fixed seed. validation: sorted by id = stable and mixed order
    df = df.sample(frac=1, random_state=seed) if shuffle else df.sort_values('id')
    return df.reset_index(drop=True)

train_pairs = finalize(gold_train, nei_train, shuffle=True, seed=SEED)
val_pairs = finalize(gold_val, nei_val, shuffle=False)
check_contract(train_pairs, 'train_pairs'); check_contract(val_pairs, 'val_pairs')
assert set(train_pairs.id).isdisjoint(set(val_pairs.id)), 'a claim id is in both train and validation'

def words(s):
    return s.str.split().str.len()

for name, df in [('train', train_pairs), ('validation', val_pairs)]:
    print(f'\n[{name}] {len(df)} claim-evidence pairs')
    t = pd.DataFrame({'claims': df.label.value_counts()})
    t['share'] = (t.claims / t.claims.sum()).round(3)
    t['mean_n_sent'] = df.groupby('label').n_sent.mean().round(2)
    t['mean_words'] = df.assign(w=words(df.evidence)).groupby('label').w.mean().round(1)   # length-shortcut check
    print(t.reindex(LABELS))
    print('evidence source:', df.evidence_source.value_counts().to_dict())

# texts that carry more than one label in the merged TRAIN table (S vs R, S vs NEI, R vs NEI)
tl = train_pairs.groupby('claim')['label'].nunique()
n_multi_text = int((tl > 1).sum())
print(f'\ntrain claim texts with more than one label: {n_multi_text} of {len(tl)}')
n_val_in_train = int(val_pairs.claim.isin(set(train_pairs.claim)).sum())
print(f'validation claims whose exact text also appears in train: {n_val_in_train} of {len(val_pairs)}')

train_pairs: contract OK (145448 rows)
val_pairs: contract OK (9997 rows)

[train] 145448 claim-evidence pairs
                 claims  share  mean_n_sent  mean_words
label                                                  
SUPPORTS          80035  0.550         1.84        57.3
REFUTES           29775  0.205         1.91        57.8
NOT ENOUGH INFO   35638  0.245         1.87        50.2
evidence source: {'gold': 109810, 'tfidf': 35638}

[validation] 9997 claim-evidence pairs
                 claims  share  mean_n_sent  mean_words
label                                                  
SUPPORTS           3333  0.333         1.66        47.4
REFUTES            3333  0.333         1.69        47.4
NOT ENOUGH INFO    3331  0.333         1.63        41.7
evidence source: {'gold': 6666, 'tfidf': 3331}

train claim texts with more than one label: 753 of 135543
validation claims whose exact text also appears in train: 1 of 9997


In [13]:
from transformers import AutoTokenizer
try:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
except Exception as e:
    raise RuntimeError(f'Could not load tokenizer {MODEL_NAME}. Check the internet connection and retry. Original error: {e}')

ex = train_pairs[train_pairs.label == 'NOT ENOUGH INFO'].iloc[0]
enc = tok(ex.claim, ex.evidence, truncation=True, max_length=MAX_LEN)
toks = tok.convert_ids_to_tokens(enc['input_ids'])
print('claim   :', ex.claim)
print('evidence:', ex.evidence[:200])
print('tokens  :', toks[:12], '...', toks[-4:])
print('length  :', len(enc['input_ids']), '| attention_mask all ones (no padding here):', set(enc['attention_mask']) == {1})

claim   : Tyra Banks had a small role in a splatter film.
evidence: The Tyra Banks Show , also known as and shortened to Tyra or The Tyra Show , is an American talk show hosted by Tyra Banks .
tokens  : ['[CLS]', 'ty', '##ra', 'banks', 'had', 'a', 'small', 'role', 'in', 'a', 'sp', '##lat'] ... ['##ra', 'banks', '.', '[SEP]']
length  : 48 | attention_mask all ones (no padding here): True


In [14]:
def token_lengths(df):
    ids = tok(df['claim'].tolist(), df['evidence'].tolist(), truncation=False)['input_ids']
    return np.array([len(x) for x in ids])

trunc_rows, trunc_summary = [], {}
def trunc_report(df, name):
    lens = token_lengths(df)
    over = pd.Series(lens > MAX_LEN, index=df.index)
    by_label = over.groupby(df['label']).mean().reindex(LABELS)
    by_src = over.groupby(df['evidence_source']).mean()
    by_n = over.groupby(pd.cut(df['n_sent'], [0, 1, 2, 3, 1000]), observed=True).mean()
    print(f'[{name}] tokens: mean={lens.mean():.1f} median={np.median(lens):.0f} p95={np.percentile(lens, 95):.0f} max={lens.max()}')
    print(f'   truncated at {MAX_LEN}: {int(over.sum()):,} of {len(df):,} ({over.mean():.1%})')
    print('   by label   :', by_label.round(3).to_dict())
    print('   by source  :', by_src.round(3).to_dict())
    print('   by n_sent  :', {str(k): round(float(v), 3) for k, v in by_n.items()})
    spread = float(by_label.max() - by_label.min())
    if spread > 0.05:
        print(f'   WARNING: truncation differs by {spread:.1%} between labels. Tell the group before training.')
    trunc_rows.append((name, 'overall', float(over.mean()), len(df)))
    for k, v in by_label.items(): trunc_rows.append((name, f'label={k}', float(v), int((df.label == k).sum())))
    for k, v in by_src.items(): trunc_rows.append((name, f'source={k}', float(v), int((df.evidence_source == k).sum())))
    trunc_summary[name] = dict(overall=float(over.mean()), by_label={k: float(v) for k, v in by_label.items()},
                               median_tokens=float(np.median(lens)), p95_tokens=float(np.percentile(lens, 95)))

trunc_report(train_pairs, 'train')
trunc_report(val_pairs, 'validation')
pd.DataFrame(trunc_rows, columns=['split', 'group', 'truncated_share', 'n']).to_csv(f'{ROOT}/results/truncation_stats{suffix}.csv', index=False)
print('saved results/truncation_stats%s.csv' % suffix)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (628 > 512). Running this sequence through the model will result in indexing errors


[train] tokens: mean=76.8 median=54 p95=200 max=1742
   truncated at 128: 14,770 of 145,448 (10.2%)
   by label   : {'SUPPORTS': 0.106, 'REFUTES': 0.11, 'NOT ENOUGH INFO': 0.084}
   by source  : {'gold': 0.107, 'tfidf': 0.084}
   by n_sent  : {'(0, 1]': 0.004, '(1, 2]': 0.052, '(2, 3]': 0.264, '(3, 1000]': 0.883}
[validation] tokens: mean=65.8 median=49 p95=159 max=1659
   truncated at 128: 718 of 9,997 (7.2%)
   by label   : {'SUPPORTS': 0.076, 'REFUTES': 0.079, 'NOT ENOUGH INFO': 0.061}
   by source  : {'gold': 0.077, 'tfidf': 0.061}
   by n_sent  : {'(0, 1]': 0.001, '(1, 2]': 0.03, '(2, 3]': 0.199, '(3, 1000]': 0.842}
saved results/truncation_stats.csv


In [15]:
from datasets import Dataset, load_from_disk

def tokenize_pairs(df):
    ds = Dataset.from_pandas(df[['claim', 'evidence', 'label_id']].rename(columns={'label_id': 'labels'}),
                             preserve_index=False)
    return ds.map(lambda b: tok(b['claim'], b['evidence'], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=['claim', 'evidence'])

train_ds, val_ds = tokenize_pairs(train_pairs), tokenize_pairs(val_pairs)
print(train_ds)
# the column must be called labels, and validation order must equal val_pairs (Member 3 relies on this)
assert len(train_ds) == len(train_pairs) and len(val_ds) == len(val_pairs)
assert (np.array(train_ds['labels']) == train_pairs.label_id.values).all()
assert (np.array(val_ds['labels']) == val_pairs.label_id.values).all(), 'validation order mismatch'

train_pairs.to_parquet(f'{P}/train_pairs{suffix}.parquet', index=False)
val_pairs.to_parquet(f'{P}/val_pairs{suffix}.parquet', index=False)
train_ds.save_to_disk(f'{P}/train_tok{suffix}')
val_ds.save_to_disk(f'{P}/val_tok{suffix}')

# reload test: what Member 3 will do
back = load_from_disk(f'{P}/val_tok{suffix}')
assert len(back) == len(val_pairs) and (np.array(back['labels']) == val_pairs.label_id.values).all()
check_contract(pd.read_parquet(f'{P}/val_pairs{suffix}.parquet'), 'val_pairs (reloaded)')
print('saved:', sorted(f for f in os.listdir(P) if ('pairs' in f or '_tok' in f)))

Map:   0%|          | 0/145448 [00:00<?, ? examples/s]

Map:   0%|          | 0/9997 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 145448
})


Saving the dataset (0/1 shards):   0%|          | 0/145448 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9997 [00:00<?, ? examples/s]

val_pairs (reloaded): contract OK (9997 rows)
saved: ['toy_pairs.parquet', 'train_pairs.parquet', 'train_pairs_mini.parquet', 'train_tok', 'train_tok_mini', 'val_pairs.parquet', 'val_pairs_mini.parquet', 'val_tok', 'val_tok_mini']


In [16]:
def audit_summary():
    p = f'{ROOT}/results/nei_audit_train.csv'
    if not os.path.exists(p):
        return {'status': 'audit sheet not found (run 04 with MINI = False)'}
    a = pd.read_csv(p).fillna('')
    cols = [c for c in a.columns if c.startswith('judgement_')]
    norm = {c: a[c].astype(str).str.strip().str.lower() for c in cols}
    filled = [c for c in cols if (norm[c] != '').sum() > 0]
    if not filled:
        return {'status': 'audit sheet exists but nobody has filled it in yet', 'n_pairs': int(len(a))}
    out = {'status': 'ok', 'n_pairs': int(len(a)), 'members_filled': len(filled), 'per_member': {}}
    for c in filled:
        m = norm[c] != ''
        out['per_member'][c] = dict(judged=int(m.sum()),
                                    not_neither=float((norm[c][m] != 'neither').mean()),
                                    counts=norm[c][m].value_counts().to_dict())
    if len(filled) >= 2:
        both = np.all([norm[c] != '' for c in filled], axis=0)
        agree = np.all([norm[c][both] == norm[filled[0]][both] for c in filled[1:]], axis=0).mean() if both.any() else float('nan')
        out['all_judged_pairs'] = int(both.sum()); out['full_agreement'] = float(agree)
    return out

audit = audit_summary()
print(json.dumps(audit, indent=2, default=str))

def describe(df):
    return dict(pairs=len(df), by_label=df.label.value_counts().to_dict(),
                share={k: round(v, 4) for k, v in df.label.value_counts(normalize=True).items()},
                mean_n_sent=round(float(df.n_sent.mean()), 3), mean_words=round(float(words(df.evidence).mean()), 1))

desc = dict(
    mode='MINI (test run, not for the report)' if MINI else 'full',
    model=MODEL_NAME, max_length=MAX_LEN,
    train=describe(train_pairs), validation=describe(val_pairs),
    drops=dict(
        gold_claims_dropped_for_missing_evidence='0 (Member 1: coverage 100%)',
        nei_train_raw=nei_total['train'], nei_train_kept=len(nei_train) + n_dropped_conflict,
        nei_val_raw=nei_total['val'], nei_val_kept=len(nei_val),
        nei_train_dropped_for_same_text_as_sr_claim=n_dropped_conflict),
    text_conflicts=dict(nei_same_text_train=int(conf_train.sum()), nei_same_text_val=int(conf_val.sum()),
                        train_texts_with_multiple_labels=n_multi_text, val_texts_also_in_train=n_val_in_train),
    truncation=trunc_summary,
    nei_method=('TF-IDF (English stop words removed, sublinear tf, min_df 2, max_df 0.5) over Wikipedia sentences of the pages '
                'cited as gold evidence, with separate pools for train and validation; the 451 pages needed by both splits are '
                'only in the validation pool. Each NEI claim copies the sentence count and word count of a random gold example; '
                'candidates are the most similar sentences, re-ranked to prefer a length close to that target; sentences under '
                '6 words are dropped. (Edit this text if the settings in 04 change.)'),
    nei_audit=audit)
with open(f'{ROOT}/results/data_description{suffix}.json', 'w') as f:
    json.dump(desc, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

L = [f'# Data description ({desc["mode"]})', '',
     f'Model: {MODEL_NAME} | max length {MAX_LEN} | validation split: paper_dev (test set never used)', '',
     f'- Train: {desc["train"]["pairs"]:,} pairs, labels {desc["train"]["by_label"]}',
     f'- Validation: {desc["validation"]["pairs"]:,} pairs, labels {desc["validation"]["by_label"]}',
     f'- Mean words per passage by split: train {desc["train"]["mean_words"]}, validation {desc["validation"]["mean_words"]}',
     f'- NEI claims: train {desc["drops"]["nei_train_raw"]} raw -> {desc["drops"]["nei_train_kept"]} built; '
     f'validation {desc["drops"]["nei_val_raw"]} raw -> {desc["drops"]["nei_val_kept"]} built',
     f'- NEI train claims dropped for sharing text with a SUPPORTS/REFUTES claim: {n_dropped_conflict} '
     f'(found {int(conf_train.sum())} in train, {int(conf_val.sum())} in validation)',
     f'- Truncated at {MAX_LEN} tokens: train {trunc_summary["train"]["overall"]:.1%}, validation {trunc_summary["validation"]["overall"]:.1%}',
     f'- NEI construction: {desc["nei_method"]}',
     f'- NEI audit: {audit.get("status")}', '']
open(f'{ROOT}/results/data_description{suffix}.md', 'w').write('\n'.join(L))
print('\n'.join(L))

{
  "status": "audit sheet exists but nobody has filled it in yet",
  "n_pairs": 50
}
# Data description (full)

Model: distilbert-base-uncased | max length 128 | validation split: paper_dev (test set never used)

- Train: 145,448 pairs, labels {'SUPPORTS': 80035, 'NOT ENOUGH INFO': 35638, 'REFUTES': 29775}
- Validation: 9,997 pairs, labels {'REFUTES': 3333, 'SUPPORTS': 3333, 'NOT ENOUGH INFO': 3331}
- Mean words per passage by split: train 55.7, validation 45.5
- NEI claims: train 35639 raw -> 35638 built; validation 3333 raw -> 3331 built
- NEI train claims dropped for sharing text with a SUPPORTS/REFUTES claim: 0 (found 1101 in train, 35 in validation)
- Truncated at 128 tokens: train 10.2%, validation 7.2%
- NEI construction: TF-IDF (English stop words removed, sublinear tf, min_df 2, max_df 0.5) over Wikipedia sentences of the pages cited as gold evidence, with separate pools for train and validation; the 451 pages needed by both splits are only in the validation pool. Each NEI cl